In [1]:
import warnings
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

df = pd.read_csv("../data/oasis_longitudinal.csv")
df["label"] = (df["CDR"] > 0).astype(int)
df["Sex"] = (df["M/F"] == "M").astype(int)

y = df["label"]
groups = df["Subject ID"]
cv = GroupKFold(n_splits=5)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

feature_sets = {
    "B (no MMSE)": ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV"],
    "A (with MMSE)": ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV", "MMSE"],
}

fill = lambda: SimpleImputer(strategy="median")
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(fill(), StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(fill(),
                                   RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0)),
    "XGBoost": make_pipeline(fill(),
                             XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.1,
                                           eval_metric="logloss", random_state=0)),
}

rows = []
for exp_name, feats in feature_sets.items():
    X = df[feats]
    for model_name, model in models.items():
        res = cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring)
        row = {"experiment": exp_name, "model": model_name}
        for m in scoring:
            row[m] = round(res["test_" + m].mean(), 2)
            row[m + "_sd"] = round(res["test_" + m].std(), 2)
        rows.append(row)

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("../report/results_v1.csv", index=False)

   experiment               model  accuracy  accuracy_sd  precision  precision_sd  recall  recall_sd   f1  f1_sd  roc_auc  roc_auc_sd
  B (no MMSE)            Baseline      0.55         0.06       0.00          0.00    0.00       0.00 0.00   0.00     0.50        0.00
  B (no MMSE) Logistic Regression      0.65         0.08       0.63          0.15    0.59       0.09 0.60   0.11     0.73        0.10
  B (no MMSE)       Random Forest      0.65         0.06       0.62          0.16    0.56       0.13 0.58   0.11     0.69        0.10
  B (no MMSE)             XGBoost      0.65         0.08       0.61          0.15    0.61       0.13 0.61   0.13     0.67        0.11
A (with MMSE)            Baseline      0.55         0.06       0.00          0.00    0.00       0.00 0.00   0.00     0.50        0.00
A (with MMSE) Logistic Regression      0.79         0.04       0.80          0.10    0.71       0.10 0.75   0.08     0.89        0.04
A (with MMSE)       Random Forest      0.78         0.05      

In [2]:
from sklearn.model_selection import GroupKFold

cv = GroupKFold(n_splits=5)
for i, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    train_people = set(groups.iloc[train_idx])
    test_people = set(groups.iloc[test_idx])
    overlap = train_people & test_people
    print(f"Round {i+1}: {len(test_people)} test people, overlap = {len(overlap)}")

Round 1: 30 test people, overlap = 0
Round 2: 30 test people, overlap = 0
Round 3: 30 test people, overlap = 0
Round 4: 30 test people, overlap = 0
Round 5: 30 test people, overlap = 0
